# 🚀 ColabMCP - ModelScope 创空间版本

这个 Notebook 用于在 ModelScope 创空间部署 MCP 服务器。

## 与 Colab 版本的区别
- ✅ 无需 ngrok，自动获得公网 URL
- ✅ 端口固定为 7860
- ✅ 更稳定，不会 90 分钟断开
- ✅ 免费资源：2vCPU / 16GB RAM

## 部署步骤
1. 在 ModelScope 创建创空间
2. 上传此 Notebook 或复制代码
3. 运行所有 cells
4. 获取自动生成的公网 URL

## 1️⃣ 安装依赖

In [ ]:
!pip install flask requests psutil -q
print("✅ 依赖安装完成!")

## 2️⃣ 定义 MCP 服务器代码

In [ ]:
%%writefile mcp_server.py
import os
import sys
import json
import time
import traceback
import subprocess
import gc
import threading
from flask import Flask, request, jsonify
import psutil

# 全局变量
runtime_variables = {}
start_time = time.time()
execution_lock = threading.Lock()

# 创建 Flask 应用
app = Flask(__name__)

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        "name": "ColabMCP Server (ModelScope)",
        "version": "1.1.0",
        "status": "running",
        "platform": "ModelScope",
        "endpoints": ["/health", "/probe", "/execute", "/variables", "/files", "/cleanup"]
    })

@app.route('/health', methods=['GET'])
def health_check():
    mem = psutil.virtual_memory()
    return jsonify({
        "status": "ok",
        "platform": "ModelScope",
        "uptime_minutes": round((time.time() - start_time) / 60, 2),
        "memory_available_gb": round(mem.available / (1024**3), 2),
        "memory_total_gb": round(mem.total / (1024**3), 2),
        "memory_used_pct": round(mem.percent, 2),
        "gpu_available": _check_gpu()
    })

@app.route('/probe', methods=['GET'])
def probe_environment():
    gpu_info = ""
    try:
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], 
                              capture_output=True, text=True, timeout=10)
        gpu_info = result.stdout
    except:
        gpu_info = "No GPU available"
    
    installed_packages = []
    try:
        result = subprocess.run(['pip', 'list', '--format=freeze'], capture_output=True, text=True, timeout=30)
        for line in result.stdout.split('\\n'):
            if '==' in line:
                installed_packages.append(line.strip())
    except:
        pass
    
    mem = psutil.virtual_memory()
    
    return jsonify({
        "gpu_info": gpu_info,
        "memory_total_gb": round(mem.total / (1024**3), 2),
        "memory_available_gb": round(mem.available / (1024**3), 2),
        "python_version": sys.version,
        "installed_packages": installed_packages[:100],
        "total_packages": len(installed_packages)
    })

@app.route('/execute', methods=['POST'])
def execute_code():
    """执行 Python 代码，带错误隔离和超时保护"""
    if not execution_lock.acquire(blocking=False):
        return jsonify({
            "success": False,
            "error": "另一个代码正在执行中，请稍后重试"
        })
    
    try:
        data = request.get_json()
        code = data.get('code', '')
        timeout = min(data.get('timeout', 300), 600)
        
        if not code:
            return jsonify({"success": False, "error": "No code provided"})
        
        exec_globals = {
            '__builtins__': __builtins__,
            **runtime_variables
        }
        exec_locals = {}
        
        from io import StringIO
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = captured_stdout = StringIO()
        sys.stderr = captured_stderr = StringIO()
        
        start_exec_time = time.time()
        
        try:
            exec(code, exec_globals, exec_locals)
            
            for key, value in exec_locals.items():
                if not key.startswith('_'):
                    try:
                        json.dumps({key: str(type(value))})
                        runtime_variables[key] = value
                    except:
                        pass
            
            stdout_output = captured_stdout.getvalue()
            stderr_output = captured_stderr.getvalue()
            
            return jsonify({
                "success": True,
                "stdout": stdout_output,
                "stderr": stderr_output,
                "execution_time_sec": round(time.time() - start_exec_time, 3),
                "variables": list(exec_locals.keys())
            })
            
        except KeyboardInterrupt:
            return jsonify({
                "success": False,
                "error": "代码被中断",
                "execution_time_sec": round(time.time() - start_exec_time, 3)
            })
        except Exception as e:
            return jsonify({
                "success": False,
                "error": str(e),
                "error_type": type(e).__name__,
                "traceback": traceback.format_exc(),
                "stdout": captured_stdout.getvalue(),
                "stderr": captured_stderr.getvalue(),
                "execution_time_sec": round(time.time() - start_exec_time, 3)
            })
        
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr
    
    except Exception as e:
        return jsonify({
            "success": False,
            "error": f"服务器内部错误: {str(e)}",
            "error_type": type(e).__name__,
            "traceback": traceback.format_exc()
        })
    
    finally:
        execution_lock.release()

@app.route('/variables', methods=['GET'])
def list_variables():
    vars_info = {}
    for key, value in runtime_variables.items():
        try:
            var_info = {"type": str(type(value).__name__)}
            if hasattr(value, 'shape'):
                var_info["shape"] = list(value.shape) if hasattr(value.shape, '__iter__') else str(value.shape)
            if hasattr(value, '__len__'):
                try:
                    var_info["length"] = len(value)
                except:
                    pass
            vars_info[key] = var_info
        except:
            vars_info[key] = {"type": str(type(value).__name__)}
    
    return jsonify({"variables": vars_info, "count": len(vars_info)})

@app.route('/files', methods=['GET'])
def list_files():
    files = []
    # ModelScope 工作目录
    work_dir = os.getcwd()
    try:
        for f in os.listdir(work_dir):
            path = os.path.join(work_dir, f)
            try:
                size = os.path.getsize(path)
                files.append({
                    "name": f,
                    "path": path,
                    "size_bytes": size,
                    "size_readable": f"{size/1024:.1f} KB" if size < 1024*1024 else f"{size/1024/1024:.1f} MB",
                    "is_dir": os.path.isdir(path)
                })
            except:
                pass
    except Exception as e:
        return jsonify({"error": str(e), "files": []})
    return jsonify({"files": files, "count": len(files)})

@app.route('/cleanup', methods=['POST'])
def cleanup():
    global runtime_variables
    runtime_variables = {}
    gc.collect()
    
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except:
        pass
    
    mem = psutil.virtual_memory()
    return jsonify({
        "success": True,
        "message": "Memory cleaned",
        "memory_available_gb": round(mem.available / (1024**3), 2)
    })

def _check_gpu():
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
        return result.returncode == 0
    except:
        return False

if __name__ == '__main__':
    print("\n" + "="*60)
    print("🚀 ColabMCP 服务器启动中 (ModelScope 版本)")
    print("="*60)
    print("端口: 7860 (ModelScope 固定端口)")
    print("版本: 1.1.0")
    print("="*60 + "\n")
    # ModelScope 固定使用 7860 端口
    app.run(port=7860, host='0.0.0.0', threaded=True)

print("✅ 服务器代码已保存到 mcp_server.py")

## 3️⃣ 启动 MCP 服务器

**重要**: ModelScope 创空间会自动提供公网 URL，无需 ngrok！

运行后，在创空间页面顶部可以看到公网访问地址。

In [ ]:
print("🚀 启动 MCP 服务器...")
print("📌 ModelScope 会自动提供公网 URL")
print("📌 端口: 7860")
print()

# 运行服务器
!python mcp_server.py

---
## 📝 使用说明

服务器启动后，ModelScope 会自动分配一个公网 URL，格式类似：
```
https://your-space-name.modelscope.cn
```

### 测试连接
```python
import requests

url = "https://your-space-name.modelscope.cn"

# 健康检查
r = requests.get(f"{url}/health")
print(r.json())
```

### 执行代码
```python
# 执行 Python 代码
r = requests.post(f"{url}/execute", json={
    "code": "print('Hello from ModelScope!')"
})
print(r.json())
```

### 执行 Shell 命令
```python
import subprocess

r = requests.post(f"{url}/execute", json={
    "code": "import subprocess; subprocess.run(['pip', 'install', 'torch'], check=True)"
})
print(r.json())
```

---
## 🔧 ModelScope vs Colab 对比

| 特性 | ModelScope | Google Colab |
|------|-----------|--------------|
| 公网 URL | ✅ 自动提供 | 需要 ngrok |
| 端口 | 7860 固定 | 任意 |
| 稳定性 | 长期运行 | 90分钟断开 |
| 免费 CPU | 2vCPU/16GB | 2vCPU/12GB |
| 免费 GPU | xGPU（申请） | T4（随机） |